# import libraries

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

In [3]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(18):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-270m-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

print(transcoder_paths)  # verify all 18 are present

{0: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_0_width_16k_l0_small/params.safetensors', 1: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_1_width_16k_l0_small/params.safetensors', 2: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_2_width_16k_l0_small/params.safetensors', 3: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_3_width_16k_l0_small/params.safetensors', 4: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_a

In [4]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-270m-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/circuit_tracer/transcoder/single_layer_transcoder.py:513: UserWarning: Lazy loading is not supported for GemmaScope2 format due to different key naming conventions. Setting lazy_encoder=False and lazy_decoder=False.
  warnings.warn(


In [5]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-270m",
    transcoders=transcoder_set,       # ← our manually loaded TranscoderSet
    backend="transformerlens",
    dtype=torch.bfloat16,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

Loaded pretrained model google/gemma-3-270m into HookedTransformer


# 1. get top influential features

In [6]:
import json
import glob
from collections import defaultdict

In [7]:
GRAPH_DIR = "./graphs/gemma-3-270m"  # ← set this
DESCRIBED_LAYERS = {5, 9, 12, 15}

In [8]:
def parse_local_feat(node):
    """Extract (layer_int, local_feat_idx) from jsNodeId."""
    js = node.get("jsNodeId", "")
    try:
        layer_feat, _ = js.rsplit("-", 1)
        layer_str, feat_str = layer_feat.split("_", 1)
        return int(layer_str), int(feat_str)
    except Exception:
        return None, None

In [9]:
all_features = defaultdict(float)
 
for fpath in sorted(glob.glob(f"{GRAPH_DIR}/step-*.json")):
    with open(fpath) as f:
        data = json.load(f)
    for node in data.get("nodes", []):
        if "transcoder" not in node.get("feature_type", ""):
            continue
        layer, local_feat = parse_local_feat(node)
        if layer is None:
            continue
        inf = abs(node.get("influence", 0) or 0)
        all_features[(layer, local_feat)] += inf

### top ten from all layers

In [10]:
top_10_overall = sorted(all_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 OVERALL (all layers)")
print("=" * 70)
for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}  {in_desc}")

TOP 10 OVERALL (all layers)
 1. L17 F 1049  influence= 11.3329    undescribed
 2. L16 F14713  influence= 11.2031    undescribed
 3. L16 F 9362  influence= 10.9282    undescribed
 4. L16 F13993  influence= 10.9144    undescribed
 5. L17 F12205  influence= 10.8671    undescribed
 6. L17 F 4359  influence= 10.8653    undescribed
 7. L16 F13891  influence= 10.7674    undescribed
 8. L16 F 5212  influence= 10.7454    undescribed
 9. L17 F 2753  influence= 10.5294    undescribed
10. L17 F 3844  influence= 10.4468    undescribed


### top ten from only described layers

In [11]:
described_features = {lf: s for lf, s in all_features.items() if lf[0] in DESCRIBED_LAYERS}
top_10_described_sorted = sorted(described_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM DESCRIBED LAYERS ONLY (5, 9, 12, 15)")
print("=" * 70)
if top_10_described_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_described_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_described = [lf for lf, _ in top_10_described_sorted]

TOP 10 FROM DESCRIBED LAYERS ONLY (5, 9, 12, 15)
 1. L15 F 2094  influence=  6.7952
 2. L15 F14225  influence=  6.6352
 3. L15 F 7651  influence=  6.3633
 4. L15 F13723  influence=  6.0249
 5. L15 F 3129  influence=  5.6426
 6. L15 F15858  influence=  5.1249
 7. L15 F12182  influence=  5.0907
 8. L15 F 7490  influence=  4.9146
 9. L15 F14104  influence=  4.5888
10. L15 F 6769  influence=  4.5676


### top ten from every other layer

In [12]:
other_features = {lf: s for lf, s in all_features.items() if lf[0] not in DESCRIBED_LAYERS}
top_10_undescribed_sorted = sorted(other_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM OTHER LAYERS (not 5, 9, 12, 15)")
print("=" * 70)
if top_10_undescribed_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_undescribed_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_undescribed = [lf for lf, _ in top_10_undescribed_sorted]

TOP 10 FROM OTHER LAYERS (not 5, 9, 12, 15)
 1. L17 F 1049  influence= 11.3329
 2. L16 F14713  influence= 11.2031
 3. L16 F 9362  influence= 10.9282
 4. L16 F13993  influence= 10.9144
 5. L17 F12205  influence= 10.8671
 6. L17 F 4359  influence= 10.8653
 7. L16 F13891  influence= 10.7674
 8. L16 F 5212  influence= 10.7454
 9. L17 F 2753  influence= 10.5294
10. L17 F 3844  influence= 10.4468


### summary

In [13]:
print(f"Total unique transcoder features: {len(all_features)}")
print(f"Features in described layers:     {len(described_features)}")
print(f"Features in other layers:         {len(other_features)}")

Total unique transcoder features: 3052
Features in described layers:     371
Features in other layers:         2681


# get clerp description for described layers (top ten only)

In [14]:
import requests
import time

def fetch_clerps_batch(features_list, model_id="gemma-3-270m", slug_suffix="gemmascope-2-res-16k", delay=0.3):
    """Fetch clerp descriptions for a list of (layer, feature_idx) tuples."""
    clerps = {}
    
    for i, (layer, feat) in enumerate(features_list):
        source = f"{layer}-{slug_suffix}"
        url = f"https://www.neuronpedia.org/api/feature/{model_id}/{source}/{feat}"
        
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                exps = r.json().get("explanations", [])
                desc = exps[0].get("description") if exps else None
                clerps[(layer, feat)] = desc
                status = "✓" if desc else "✗ (no desc)"
            else:
                clerps[(layer, feat)] = None
                status = f"✗ (HTTP {r.status_code})"
        except Exception as e:
            clerps[(layer, feat)] = None
            status = f"✗ ({str(e)[:30]})"
        
        print(f"  [{i+1}/{len(features_list)}] L{layer:2d} F{feat:5d}: {status}")
        time.sleep(delay)
    
    return clerps

# Extract the top 10 described features
features_to_fetch = [lf for lf, _ in top_10_described_sorted]

# Fetch clerps
print("Fetching clerps for top 10 described features...")
clerps = fetch_clerps_batch(features_to_fetch)

# Print results
print("\n" + "="*70)
print("TOP 10 DESCRIBED FEATURES WITH CLERPS")
print("="*70)
for (layer, feat), score in top_10_described_sorted:
    desc = clerps.get((layer, feat))
    print(f"L{layer:2d} F{feat:5d} (influence={score:8.4f}): {desc or '[no description found]'}")

Fetching clerps for top 10 described features...


  [1/10] L15 F 2094: ✓
  [2/10] L15 F14225: ✓
  [3/10] L15 F 7651: ✓
  [4/10] L15 F13723: ✓
  [5/10] L15 F 3129: ✓
  [6/10] L15 F15858: ✓
  [7/10] L15 F12182: ✓
  [8/10] L15 F 7490: ✓
  [9/10] L15 F14104: ✓
  [10/10] L15 F 6769: ✓

TOP 10 DESCRIBED FEATURES WITH CLERPS
L15 F 2094 (influence=  6.7952): keywords, words, names
L15 F14225 (influence=  6.6352): words starting with "re"
L15 F 7651 (influence=  6.3633): vast plains, lush forests
L15 F13723 (influence=  6.0249): is a or is equivalent
L15 F 3129 (influence=  5.6426): code generation and summarization
L15 F15858 (influence=  5.1249): describing rooms and places
L15 F12182 (influence=  5.0907): We presenting research
L15 F 7490 (influence=  4.9146): all of
L15 F14104 (influence=  4.5888): UK, Australia, government, public
L15 F 6769 (influence=  4.5676): question about complexity


# intervention

### all 20 features suppressed

In [15]:
# Your prompt
prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

# Target position (typically -1 for last token, or specify the rhyme token position)
TARGET_POS = -1

In [16]:
all_features = top_10_described + top_10_undescribed

intervention_tuples = [
    (layer, TARGET_POS, feat, 0.0)
    for layer, feat in all_features
]

print("=" * 70)
print("INTERVENTION TUPLES (all 20 features, suppressed at target position)")
print("=" * 70)
for i, (layer, pos, feat, val) in enumerate(intervention_tuples, 1):
    print(f"{i:2d}. Layer {layer:2d}, Position {pos:3d}, Feature {feat:5d}, Value {val}")

INTERVENTION TUPLES (all 20 features, suppressed at target position)
 1. Layer 15, Position  -1, Feature  2094, Value 0.0
 2. Layer 15, Position  -1, Feature 14225, Value 0.0
 3. Layer 15, Position  -1, Feature  7651, Value 0.0
 4. Layer 15, Position  -1, Feature 13723, Value 0.0
 5. Layer 15, Position  -1, Feature  3129, Value 0.0
 6. Layer 15, Position  -1, Feature 15858, Value 0.0
 7. Layer 15, Position  -1, Feature 12182, Value 0.0
 8. Layer 15, Position  -1, Feature  7490, Value 0.0
 9. Layer 15, Position  -1, Feature 14104, Value 0.0
10. Layer 15, Position  -1, Feature  6769, Value 0.0
11. Layer 17, Position  -1, Feature  1049, Value 0.0
12. Layer 16, Position  -1, Feature 14713, Value 0.0
13. Layer 16, Position  -1, Feature  9362, Value 0.0
14. Layer 16, Position  -1, Feature 13993, Value 0.0
15. Layer 17, Position  -1, Feature 12205, Value 0.0
16. Layer 17, Position  -1, Feature  4359, Value 0.0
17. Layer 16, Position  -1, Feature 13891, Value 0.0
18. Layer 16, Position  -1, Fe

In [17]:
print("\n" + "=" * 70)
print("GENERATING WITH ORIGINAL FEATURES (no intervention)")
print("=" * 70)

pre_intervention_text = model.feature_intervention_generate(
    prompt,
    [],  # empty list = no interventions
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{pre_intervention_text}")

# ─────────────────────────────────────────────────────────────
# Generate with all 20 features suppressed
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("GENERATING WITH ALL 20 FEATURES SUPPRESSED")
print("=" * 70)

post_intervention_text = model.feature_intervention_generate(
    prompt,
    intervention_tuples,
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{post_intervention_text}")


GENERATING WITH ORIGINAL FEATURES (no intervention)


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
He saw a carrot and had to grab it,

GENERATING WITH ALL 20 FEATURES SUPPRESSED


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
Plotting a carrot and a carrot
He saw


In [18]:
print("\n" + "=" * 70)
print("COMPARISON: PRE vs POST INTERVENTION")
print("=" * 70)

from circuit_tracer.utils.demo_utils import display_generations_comparison

display_generations_comparison(
    prompt,
    [pre_intervention_text],
    [post_intervention_text]
)


COMPARISON: PRE vs POST INTERVENTION


### suppressed one at a time (top five)

In [19]:
print("Suppressing one feature at a time to isolate effects...")

ablation_results = []

for layer, feat in all_features[:5]:  # just first 5 for speed
    single_intervention = [(layer, TARGET_POS, feat, 0.0)]
    
    try:
        ablated_text = model.feature_intervention_generate(
            prompt,
            single_intervention,
            do_sample=False,
            verbose=False
        )[0]
        
        # Check if output changed
        changed = ablated_text != pre_intervention_text
        status = "CHANGED" if changed else "unchanged"
        
        # Get clerp if available
        clerp_desc = clerps.get((layer, feat)) if (layer, feat) in clerps else None
        clerp_str = f" | {clerp_desc}" if clerp_desc else ""
        
        ablation_results.append({
            'layer': layer,
            'feat': feat,
            'changed': changed,
            'text': ablated_text,
            'clerp': clerp_desc
        })
        
        print(f"\nL{layer:2d} F{feat:5d}: {status}{clerp_str}")
        if changed:
            print(f"  → {ablated_text}")
    
    except Exception as e:
        print(f"L{layer:2d} F{feat:5d}: ERROR ({str(e)[:50]})")

Suppressing one feature at a time to isolate effects...



L15 F 2094: CHANGED | keywords, words, names
  → A rhyming couplet:
He saw a carrot and had to grab it,
-
He saw a carrot and had to grab

L15 F14225: CHANGED | words starting with "re"
  → A rhyming couplet:
He saw a carrot and had to grab it,
*
He saw a carrot and had to grab

L15 F 7651: CHANGED | vast plains, lush forests
  → A rhyming couplet:
He saw a carrot and had to grab it,
 crédit,
He saw a carrot and had

L15 F13723: CHANGED | is a or is equivalent
  → A rhyming couplet:
He saw a carrot and had to grab it,
ings,
He saw a carrot and had to

L15 F 3129: CHANGED | code generation and summarization
  → A rhyming couplet:
He saw a carrot and had to grab it,
PLICATIONS
A rhyming couplet:
A
